In [1]:
import json
from datetime import datetime

def save_signal_to_json(signal, filename="signal_log.json"):
    data = {
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "signal": signal
    }
    with open(filename, "w") as f:
        json.dump(data, f, indent=4)
    print(f"[JSON] Signal saved: {data}")

# Example usage:
signal = True  # or False
save_signal_to_json(signal)


[JSON] Signal saved: {'timestamp': '2025-07-02 05:54:21', 'signal': True}


In [2]:
!pip install flask


  Attempting uninstall: click
    Found existing installation: click 7.1.2
    Uninstalling click-7.1.2:
      Successfully uninstalled click-7.1.2


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
typer 0.1.1 requires click<7.2.0,>=7.1.1, but you have click 8.1.8 which is incompatible.


In [3]:
from flask import Flask, jsonify, request, abort
import json

API_KEY = "mysecretkey"  # change this

app = Flask(__name__)

@app.route("/signal", methods=["GET"])
def get_signal():
    key = request.args.get("key")
    if key != API_KEY:
        abort(403)
    with open("signal_log.json", "r") as f:
        data = json.load(f)
    return jsonify(data)


In [ ]:
from flask import Flask, jsonify
import json

app = Flask(__name__)

@app.route("/")
def root():
    return "Signal API running. Go to /signal to get the latest value."

@app.route("/signal", methods=["GET"])
def get_signal():
    with open("signal_log.json", "r") as f:
        data = json.load(f)
    return jsonify(data)

if __name__ == "__main__":
    app.run(host="127.0.0.1", port=8080)


 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:8080
Press CTRL+C to quit
127.0.0.1 - - [02/Jul/2025 16:44:16] "GET /signal HTTP/1.1" 200 -
127.0.0.1 - - [02/Jul/2025 16:44:16] "GET /favicon.ico HTTP/1.1" 404 -
127.0.0.1 - - [02/Jul/2025 16:56:10] "GET /signal HTTP/1.1" 200 -


In [ ]:
class IBTradingBot:
    def __init__(self, usd_amount=150.0):
        self.usd_amount = usd_amount
        self.ib = IBApi()
        self.clientId = 1
        self.port = 7496
        self.host = "127.0.0.1"
        self.pos = 0
        self.ticker = None

    def connect(self, clientId=1):
        self.ib.connect(self.host, self.port, clientId)
        threading.Thread(target=self.ib.run, daemon=True).start()
        if not self.ib.connected_evt.wait(5):
            raise RuntimeError("No nextValidId")

    def disconnect(self):
        self.ib.disconnect()

    def make_contract(self, symbol):
        c = Contract()
        c.symbol, c.secType, c.exchange, c.primaryExchange, c.currency = symbol, "STK", "SMART", "ISLAND", "USD"
        return c

    def place_market_order(self, symbol, action):
        price = 100
        qty = max(int(self.usd_amount // price), 1)
        oid = self.ib.nextOrderId
        self.ib.nextOrderId += 1
        self.ib.placeOrder(oid, self.make_contract(symbol), Order(action=action, orderType="MKT", totalQuantity=qty, tif="DAY", transmit=True))
        self.pos = 1 if action == "BUY" else 0

    def buy(self, s): self.place_market_order(s, "BUY")
    def close_buy(self, s): self.place_market_order(s, "SELL")

    def request_account_value(self):
        self.ib.reqAccountSummary(9001, "All", "NetLiquidation")
        time.sleep(2)
        self.ib.cancelAccountSummary(9001)

    def request_executions(self):
        f = ExecutionFilter()
        f.time = (datetime.datetime.now() - datetime.timedelta(days=360)).strftime("%Y%m%d-%H:%M:%S")
        self.ib.reqExecutions(9002, f)
        time.sleep(2)

    def fetch_ticker_info(self, ticker):
        tk = yf.Ticker(ticker)
        info = tk.info
        return {
            "Price": info.get("regularMarketPrice", "N/A"),
            "Bid": info.get("bid", "N/A"),
            "Ask": info.get("ask", "N/A"),
            "EPS": info.get("trailingEps", "N/A"),
            "Market Cap": info.get("marketCap", "N/A"),
            "P/E": info.get("trailingPE", "N/A"),
            "Sector": info.get("sector", "N/A"),
            "Industry": info.get("industry", "N/A")
        }

    def generate_eod_report(self, filename="eod_report.pdf", ticker="AAPL"):
        self.request_account_value()
        self.request_executions()
        df = pd.DataFrame(self.ib.account_values, columns=["Timestamp","NetLiquidation"])
        df["Timestamp"] = pd.to_datetime(df["Timestamp"])
        df["Shift"] = df["NetLiquidation"].shift(1)
        df2 = df[df["NetLiquidation"] != df["Shift"]].copy()
        plt.figure(figsize=(10,4))
        if len(df2) > 1:
            plt.plot(df2["Timestamp"], df2["NetLiquidation"], marker="o", color="blue")
        else:
            val = df["NetLiquidation"].iloc[-1]
            plt.axhline(val, linestyle="--", color="gray")
        plt.title("Net Liquidation")
        plt.xlabel("Time"); plt.ylabel("USD"); plt.grid(True)
        plt.tight_layout()
        plt.savefig("equity.png"); plt.close()

        info = self.fetch_ticker_info(ticker)
        pdf = FPDF()
        pdf.add_page()
        pdf.set_font("Arial","B",16); pdf.cell(0,10,"EOD Report",ln=True)
        pdf.set_font("Arial","",12); pdf.cell(0,10,f"Generated: {datetime.datetime.now():%Y-%m-%d %H:%M:%S}",ln=True)
        pdf.ln(5)
        pdf.image("equity.png", w=120)

        # Ticker info on right
        pdf.set_xy(135, 30)
        pdf.set_font("Arial","B",12); pdf.cell(0,8,f"Ticker: {ticker}", ln=True)
        pdf.set_font("Arial","",11)
        for k,v in info.items():
            pdf.set_x(135); pdf.cell(0,6,f"{k}: {v}", ln=True)
        pdf.ln(10)

        # Executions table
        pdf.set_font("Arial","B",14); pdf.cell(0,10,"Executions :",ln=True)
        df_exec = pd.DataFrame(self.ib.exec_details)
        if "time" in df_exec.columns:
            df_exec["time"] = pd.to_datetime(df_exec["time"], errors="coerce")
            df_exec = df_exec.sort_values("time", ascending=False)
        pdf.set_font("Arial","",10)
        col_w = [20,20,20,25,60]
        for h in ["symbol","action","qty","price","time"]:
            pdf.cell(col_w[["symbol","action","qty","price","time"].index(h)], 8, h.title(), border=1)
        pdf.ln()
        for _,r in df_exec.iterrows():
            pdf.cell(col_w[0],8,r.symbol,border=1)
            pdf.cell(col_w[1],8,r.action,border=1)
            pdf.cell(col_w[2],8,str(r.qty),border=1)
            pdf.cell(col_w[3],8,f"{r.price:.2f}",border=1)
            pdf.cell(col_w[4],8,str(r.time)[:19],border=1)
            pdf.ln()

        pdf.output(filename)
        os.remove("equity.png")
        print(f"[PDF] Saved {filename}")

In [ ]:

def is_market_open():
    """Check if the market is currently open using Polygon's market status endpoint."""
    api_key = "6FScTtPXo3hC4lyzkRb29mNJOOmLYwYF"
    url = f"https://api.polygon.io/v1/marketstatus/now?apiKey={api_key}"
    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()
        return data.get("market", "closed") == "open"
    return False
if __name__ == "__main__":
    pos = 0
    this_is_a_test = False
    first_pred = False
    clientId = 1
    connected = False

    while not connected:
        try:
            bot.connect(clientId=clientId)
            connected = True 
        except RuntimeError as e:
            if "nextValidId" in str(e):
                print(f"[WARN] nextValidId not received for clientId={clientId}, retrying...")
                bot.disconnect()
                time.sleep(2)
                clientId += 1
            else:
                raise
    try:
        while True:
            if signal and pos == 0:
                    print("Buy signal detected")
                    bot.buy(ticker)
                    pos = 1
                elif signal == False and pos == 1:
                    print("Sell signal detected")
                    bot.close_buy(ticker)
                    pos = 0
            

In [ ]:
import requests
import time
import pandas as pd
from datetime import datetime
import datetime
class IBTradingBot:
    def __init__(self, usd_amount=150.0):
        self.usd_amount = usd_amount
        self.ib = IBApi()
        self.clientId = 1
        self.port = 7496
        self.host = "127.0.0.1"
        self.pos = 0
        self.ticker = None

    def connect(self, clientId=1):
        self.ib.connect(self.host, self.port, clientId)
        threading.Thread(target=self.ib.run, daemon=True).start()
        if not self.ib.connected_evt.wait(5):
            raise RuntimeError("No nextValidId")

    def disconnect(self):
        self.ib.disconnect()

    def make_contract(self, symbol):
        c = Contract()
        c.symbol, c.secType, c.exchange, c.primaryExchange, c.currency = symbol, "STK", "SMART", "ISLAND", "USD"
        return c

    def place_market_order(self, symbol, action):
        price = 100
        qty = max(int(self.usd_amount // price), 1)
        oid = self.ib.nextOrderId
        self.ib.nextOrderId += 1
        self.ib.placeOrder(oid, self.make_contract(symbol), Order(action=action, orderType="MKT", totalQuantity=qty, tif="DAY", transmit=True))
        self.pos = 1 if action == "BUY" else 0

    def buy(self, s): self.place_market_order(s, "BUY")
    def close_buy(self, s): self.place_market_order(s, "SELL")

    def request_account_value(self):
        self.ib.reqAccountSummary(9001, "All", "NetLiquidation")
        time.sleep(2)
        self.ib.cancelAccountSummary(9001)

    def request_executions(self):
        f = ExecutionFilter()
        f.time = (datetime.datetime.now() - datetime.timedelta(days=360)).strftime("%Y%m%d-%H:%M:%S")
        self.ib.reqExecutions(9002, f)
        time.sleep(2)

    def fetch_ticker_info(self, ticker):
        tk = yf.Ticker(ticker)
        info = tk.info
        return {
            "Price": info.get("regularMarketPrice", "N/A"),
            "Bid": info.get("bid", "N/A"),
            "Ask": info.get("ask", "N/A"),
            "EPS": info.get("trailingEps", "N/A"),
            "Market Cap": info.get("marketCap", "N/A"),
            "P/E": info.get("trailingPE", "N/A"),
            "Sector": info.get("sector", "N/A"),
            "Industry": info.get("industry", "N/A")
        }

    def generate_eod_report(self, filename="eod_report.pdf", ticker="AAPL"):
        self.request_account_value()
        self.request_executions()
        df = pd.DataFrame(self.ib.account_values, columns=["Timestamp","NetLiquidation"])
        df["Timestamp"] = pd.to_datetime(df["Timestamp"])
        df["Shift"] = df["NetLiquidation"].shift(1)
        df2 = df[df["NetLiquidation"] != df["Shift"]].copy()
        plt.figure(figsize=(10,4))
        if len(df2) > 1:
            plt.plot(df2["Timestamp"], df2["NetLiquidation"], marker="o", color="blue")
        else:
            val = df["NetLiquidation"].iloc[-1]
            plt.axhline(val, linestyle="--", color="gray")
        plt.title("Net Liquidation")
        plt.xlabel("Time"); plt.ylabel("USD"); plt.grid(True)
        plt.tight_layout()
        plt.savefig("equity.png"); plt.close()

        info = self.fetch_ticker_info(ticker)
        pdf = FPDF()
        pdf.add_page()
        pdf.set_font("Arial","B",16); pdf.cell(0,10,"EOD Report",ln=True)
        pdf.set_font("Arial","",12); pdf.cell(0,10,f"Generated: {datetime.datetime.now():%Y-%m-%d %H:%M:%S}",ln=True)
        pdf.ln(5)
        pdf.image("equity.png", w=120)

        # Ticker info on right
        pdf.set_xy(135, 30)
        pdf.set_font("Arial","B",12); pdf.cell(0,8,f"Ticker: {ticker}", ln=True)
        pdf.set_font("Arial","",11)
        for k,v in info.items():
            pdf.set_x(135); pdf.cell(0,6,f"{k}: {v}", ln=True)
        pdf.ln(10)

        # Executions table
        pdf.set_font("Arial","B",14); pdf.cell(0,10,"Executions :",ln=True)
        df_exec = pd.DataFrame(self.ib.exec_details)
        if "time" in df_exec.columns:
            df_exec["time"] = pd.to_datetime(df_exec["time"], errors="coerce")
            df_exec = df_exec.sort_values("time", ascending=False)
        pdf.set_font("Arial","",10)
        col_w = [20,20,20,25,60]
        for h in ["symbol","action","qty","price","time"]:
            pdf.cell(col_w[["symbol","action","qty","price","time"].index(h)], 8, h.title(), border=1)
        pdf.ln()
        for _,r in df_exec.iterrows():
            pdf.cell(col_w[0],8,r.symbol,border=1)
            pdf.cell(col_w[1],8,r.action,border=1)
            pdf.cell(col_w[2],8,str(r.qty),border=1)
            pdf.cell(col_w[3],8,f"{r.price:.2f}",border=1)
            pdf.cell(col_w[4],8,str(r.time)[:19],border=1)
            pdf.ln()

        pdf.output(filename)
        os.remove("equity.png")
        print(f"[PDF] Saved {filename}")
API_URL = "https://3c33-2a02-2f04-8307-6e00-5094-33f9-aac8-2fac.ngrok-free.app/signal"

def is_market_open():
    """Check if the market is currently open using Polygon's market status endpoint."""
    api_key = "6FScTtPXo3hC4lyzkRb29mNJOOmLYwYF"
    url = f"https://api.polygon.io/v1/marketstatus/now?apiKey={api_key}"
    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()
        return data.get("market", "closed") == "open"
    return False
if __name__ == "__main__":
    pos = 0
    this_is_a_test = False
    first_pred = False
    clientId = 1
    connected = False

    while not connected:
        try:
            bot.connect(clientId=clientId)
            connected = True 
        except RuntimeError as e:
            if "nextValidId" in str(e):
                print(f"[WARN] nextValidId not received for clientId={clientId}, retrying...")
                bot.disconnect()
                time.sleep(2)
                clientId += 1
            else:
                raise
    try:
        while True:
            if signal and pos == 0:
                    print("Buy signal detected")
                    bot.buy(ticker)
                    pos = 1
                elif signal == False and pos == 1:
                    print("Sell signal detected")
                    bot.close_buy(ticker)
                    pos = 0
            
if __name__ == "__main__":
    pos = 0
    ticker = "CCL"  # You can assign your best.Symbol here
    bot = IBTradingBot(usd_amount=30.0)
    results_df = pd.DataFrame(columns=["timestamp", "Buy_Signal"])

    clientId = 1
    connected = False

    while not connected:
        try:
            bot.connect(clientId=clientId)
            connected = True 
        except RuntimeError as e:
            if "nextValidId" in str(e):
                print(f"[WARN] nextValidId not received for clientId={clientId}, retrying...")
                bot.disconnect()
                time.sleep(2)
                clientId += 1
            else:
                raise

    try:
        while True:
            if is_market_open():
                try:
                    response = requests.get(API_URL)
                    data = response.json()
                    signal = data.get("signal")
                    timestamp = data.get("timestamp")
                    
                    if signal is None:
                        print("[WARN] No signal in API response.")
                        continue
                    
                    print(f"[API] Signal: {signal}, Timestamp: {timestamp}")

                    if signal and pos == 0:
                        print("Buy signal detected")
                        bot.buy(ticker)
                        pos = 1
                    elif not signal and pos == 1:
                        print("Sell signal detected")
                        bot.close_buy(ticker)
                        pos = 0

                    results_df = pd.concat([
                        results_df,
                        pd.DataFrame({"timestamp": [timestamp], "Buy_Signal": [signal]})
                    ], ignore_index=True)

                except Exception as api_error:
                    print(f"[ERROR] Failed to fetch or parse signal: {api_error}")

                time.sleep(150)  # 2.5 minutes
            else:
                print("Market closed. Waiting...")
                time.sleep(300)  # Wait longer if market is closed

    except Exception as e:
        print(f"[FATAL ERROR] {e}")
    finally:
        today_str = datetime.now().strftime('%Y-%m-%d')
        results_df.to_csv(f"signal_log_{today_str}.csv", index=False)
        bot.generate_eod_report(f"eod_{today_str}.pdf", ticker=ticker)
        bot.disconnect()
